[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GarretOS/python-ai-foundations/blob/main/projects/news-insight-analyzer/news_insight_analyzer.ipynb)

# 📰 News Insight Analyzer

This notebook teaches and demonstrates a small news analyzer inspired by the Towards AI **News Analyzer: Summarize, Sentiment, and Tags** lesson. It uses Google's Gemini API.

## 🎯 Project Overview

The application sends pasted article text to Gemini and displays a 2-3 sentence summary, one sentiment label, three to five tags, and a concise key takeaway. The takeaway is the project's small original enhancement.

## 🔐 Add Your Gemini API Key Safely

In Google Colab, open the Secrets panel, add a secret named `GEMINI_API_KEY`, and enable notebook access. The next cell retrieves it at runtime. Do not type an API key directly into a visible cell or save it in the notebook.

In [ ]:
!pip install -q "google-genai>=1.0.0" "gradio>=5.0,<7.0"

import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("Gemini API key loaded from Colab Secrets.")

## 📦 Import Modules and Build the Prompt

`google-genai` uses `genai.Client()` and reads `GEMINI_API_KEY` from the environment. The prompt asks for JSON only so Python can parse the response with the standard-library `json` module.

In [ ]:
import json

import gradio as gr
from google import genai

MODEL_NAME = "gemini-3.8-flash"

def build_prompt(article_text):
    return f'''Analyze the supplied news article.

Return valid JSON only. Do not use Markdown code fences, and do not add any
prefix or suffix text. Use only these keys: summary, sentiment, tags,
key_takeaway.

Requirements:
- summary must be a concise 2-3 sentence overview.
- sentiment must be exactly positive, negative, or neutral.
- tags must be 3-5 relevant lowercase keywords or topics as a JSON list.
- key_takeaway must be one concise sentence stating the most important point.

Example: {"summary": "A concise overview.", "sentiment": "neutral",
"tags": ["technology", "policy"], "key_takeaway": "The main point."}

News article:
{article_text}'''

## 🧠 Analyze and Parse the Response

The response flow is: generated text → `json.loads()` → Python dictionary → `.get()` for each field. Tags stay as a list internally.

In [ ]:
def fallback_result(message):
    return message, "No sentiment found.", [], "No takeaway found."


def analyze_news_article(article_text):
    if not article_text or not article_text.strip():
        return fallback_result("Please paste a news article before analyzing it.")

    try:
        client = genai.Client()
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=build_prompt(article_text.strip()),
        )
        article_data = json.loads(response.text)

        summary = article_data.get("summary", "No summary found.")
        sentiment = article_data.get("sentiment", "No sentiment found.")
        tags = article_data.get("tags", [])
        key_takeaway = article_data.get("key_takeaway", "No takeaway found.")

        if sentiment not in {"positive", "negative", "neutral"}:
            sentiment = "No sentiment found."
        return summary, sentiment, tags, key_takeaway
    except json.JSONDecodeError:
        return fallback_result("Gemini returned text that was not valid JSON.")
    except Exception as error:
        print(f"News analysis failed: {error}")
        return fallback_result("The article could not be analyzed. Check your Gemini API key and try again.")

## 🏷️ Format Tags for Display

The model result keeps `tags` as a Python list. Only the display function turns that list into comma-separated text. An empty list gets a friendly display message.

In [ ]:
def format_tags(tags):
    if tags:
        return ", ".join(tags)
    return "No tags found."

print(format_tags(["ai", "technology", "policy"]))
print(format_tags([]))

## 🧩 Build the Gradio Interface

The button calls `gradio_interface()`, which prepares all four values for user-facing output.

In [ ]:
def gradio_interface(article_text):
    summary, sentiment, tags, key_takeaway = analyze_news_article(article_text)
    return summary, sentiment, format_tags(tags), key_takeaway

with gr.Blocks() as demo:
    gr.Markdown(
        "# 📰 News Insight Analyzer\nPaste a news article to get an analysis."
    )
    article_input = gr.Textbox(
        label="Paste your news article here",
        placeholder="Paste the full text of a news article...",
        lines=12,
    )
    analyze_button = gr.Button("Analyze Article")
    summary_output = gr.Textbox(label="Summary", lines=3)
    sentiment_output = gr.Textbox(label="Sentiment")
    tags_output = gr.Textbox(label="Tags")
    takeaway_output = gr.Textbox(label="Key Takeaway", lines=2)

    analyze_button.click(
        gradio_interface,
        article_input,
        [summary_output, sentiment_output, tags_output, takeaway_output],
    )

print("The Gradio interface was constructed. Run the local script or call demo.launch() to start it.")

## 🧪 Try It Yourself

Run these small experiments after adding your own `GEMINI_API_KEY` in Colab Secrets.

### Experiment 1 — Analyze a Different Article

Replace the sample text with another article, then run the cell.

In [ ]:
sample_article = "The city opened a new community garden downtown, giving residents more space to grow food and meet their neighbors. Local organizers say the project will make the area greener and more connected."

summary, sentiment, tags, key_takeaway = analyze_news_article(sample_article)
print("Summary:", summary)
print("Sentiment:", sentiment)
print("Tags:", tags)
print("Key Takeaway:", key_takeaway)

### Experiment 2 — Compare Article Tone

These two articles describe different tones. The result describes the tone of the supplied text rather than objectively judging the underlying event.

In [ ]:
positive_article = "The school celebrated a record number of student scholarships, giving more graduates the chance to continue their education."
negative_article = "The storm damaged several neighborhoods, leaving families without power and forcing the cancellation of local events."

positive_result = analyze_news_article(positive_article)
negative_result = analyze_news_article(negative_article)

print("Positive-toned article sentiment:", positive_result[1])
print("Negative-toned article sentiment:", negative_result[1])

### Experiment 3 — Empty Input

An empty article is handled before the Gemini request is made.

In [ ]:
empty_result = analyze_news_article("")
print("Summary:", empty_result[0])
print("Sentiment:", empty_result[1])
print("Tags:", empty_result[2])
print("Key Takeaway:", empty_result[3])

## 📚 What I Learned

I practiced using an environment variable for a secret, calling the current Gemini SDK, writing a focused JSON prompt, parsing a dictionary with `json.loads()`, validating simple values, and connecting Python functions to Gradio outputs.

## 📝 Notes

- Never commit a real API key, a visible key in a notebook cell, or a credential-filled `.env` file.
- Gemini output can vary, so the application includes simple fallback messages.
- The notebook constructs the interface but does not launch a public share URL automatically.